In [1]:
## By now, you know that the purpose of SCOPE is to handle complex computational tasks of large datasets.
## Handling large datasets is complex. Not necessarily because there are many computations, but because the USERS are not (yet) machines. 
## We change plans and we make mistakes. Indeed, even machines sometimes do not work properly. 
## In this tutorial I will to show how 'FLEXIBLE' is SCOPE in terms of handling the ACTIONS of USERS

In [ ]:
## The example below serves to illustrate how SCOPE must be executed. Imagine you want to run 2 jobs (job1, job2), 
## in which job2 can only start when job1 finishes. You can do so with the following command:

## scope_run_job -n ../scope_env.npy -s ABITEM/ABITEM.npy -j job1.job job2.job

## Case 1) The first time you execute this command, if everything goes well, Scope will submit the computation(s) associated to job1 to the cluster, and won't do anything else. The System file will be updated
## Case 2) If you execute once again while job1's computations are still running, nothing will happen. The System file won't be updated
## Case 3) Once job1's computations finish, if you execute, Scope will register job1, and submit job2. The System file will be updated
## Case 4) When job2's computations finish, if you execute, Scope will skip job1 (it is already registered), register job2, and the workflow will be over. The System file will be updated, and you can then analyse the results. 

# <div style="text-align: center"> Tutorial 8: How to Run SCOPE, Part 3</div>

# On the use of SCOPE (<span style="color:red">**IMPORTANT**</span>):

SCOPE works with:   

- SYSTEM-class Objects stored in binary files (the SYSTEM files)  
- plain text JOB FILES (e.g. test_job.job in Tutorial 4)   
- COMPUTATION files (i.e. input, output, and submission)  

#### - **System Files**: 

- <span style="color:green">Can be deleted or restarted</span> harmlessly as long as COMPUTATION files remain in place (i.e. same path, same filename).   
- If you move them, preserve the env.sys_path/system_name/system_name.npy structure, and run sys.set_paths()

#### - **Job Files**: 

- You can modify the *&environment* and *&options* sections anytime during Cases 1-4 above
- <span style="color:red">AVOID</span> modifying the *&job_data* and *&qc_data* sections as much as possible. 

#### - **Computation files**:
- <span style="color:red">Do not</span> remove those files unless you're really sure what you're doing, even if the results have been registered already
- If you move them, preserve the env.calcs_path/system_name/branch_name/ structure, and change the env.calcs_path in the environment  
  Otherwise, SCOPE will re-submit these very same computations, wasting computational resources (!)

This is why the ENVIRONMENT-class objects exist, and are central to the whole machinery of SCOPE. They somehow force the USER to define paths, and stick to those during the project. 

## Moving between computers: 

Let's say you have a local computer ("LOCAL") and two computational HPC clusters ("HPC1" and "HPC2") at your disposal. You can move files freely between these computers. Do the following:
1) Create an environment in each computer. That is, run "scope_config" in LOCAL, HPC1 and HPC2
2) Ideally, keep all relevant scope folders syncronized. For each environment, these are env.sources_path, env.calcs_path, env.sys_path. 
   In practice, you only need to syncronize the folders of the systems you'll work with during a given session. 

Eventually, I'd like to implement a protocol to use only LOCAL to store files, and only connect to HPC clusters to submit and retrieve the calculation files.

# Example

In [3]:
import os
import scope
from scope.read_write import *

In [4]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data"
data_folder = os.path.abspath('.')+'/Data/'
## Loads the System object from a binary file, provided in the tutorial folder
sys = load_binary(f"{data_folder}ABITEM.npy")

## Navigating the Workflow 

In [5]:
## We will discuss the B3LYP optimization job stored in our example system file. First, we navigate the workflow as discussed in Tutorial 1 (System -> Branch -> Workflow -> Job)
found, this_branch   = sys.find_branch("Isolated")
found, this_workflow = this_branch.find_workflow("ref_hs_mol")
found, this_job      = this_workflow.find_job("b3lyp_opt")
## We will replicate the steps undergone by "scope_run_job" when running this Job.
print(this_job)

---------------------------------------------------
   >>> >>> >>> JOB                                 
---------------------------------------------------
 Source Type           = specie
 Source Name           = ref_hs_mol
 Branch Name           = Isolated
 Workflow Name         = ref_hs_mol
---------------------------------------------------
 Job path              = /Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Calcs/ABITEM/Isolated/
 Job keyword           = b3lyp_opt
 Job hierarchy         = 2
 Job requisites        = ['pbe_opt']
 Job constrains        = ['self']
 Job setup             = regular
 Num Computations      = 1
----------------------------------------------------
 self.isregistered (Temp) = True
 self.isgood       (Temp) = True
 self.isfinished   (Temp) = True




## Function run_job

In [6]:
## "scope_run_job" is the CLI form to just call the function "run_job", which can be found in scope/utils. Here are the docs:
import scope.utils 
help(scope.utils.run_job)

Help on function run_job in module scope.utils.run_job:

run_job(sys_path: str, job_paths: list, global_env: str | object, handle_errors: bool = False, debug: int = 0)
    Runs a SCOPE Task (defined in the job_paths file) on a SCOPE system (in the sys_path). 
    The Configuration of the Computer is read from the GLOBAL_ENVIRONMENT, which must be configured before and given as a binary.
    This function performs the following steps:
    - Verifies the existence of the system and job files.
    - Reads input data from the job file, including environment, options, job data, and QC_data.
    - Loads the system object from the binary file and updates paths if necessary. Handy if registration and execution of tasks are performed in different computers
    - Finds or creates the required branch and workflow in the system.
    - Finds or creates the job, and checks for input changes.
    - Validates job requisites and continues only if they are fulfilled.
    - Sets up computations for the j

In [7]:
## If you open the function with a text editor, you'll identify commented blocks mentioning the STEPs

### Step 0:

In [8]:
## In Step 0, the function checks the existence of the files, and loads them. So nothing worth discussing here :)

### Step 1:

In [9]:
## In Step 1, it initiates the 'ith' LOOP for every job specified by the user. Notice that in the argument -j when calling "scope_run_job" you can specify many job_files if you wish. 
## Scope will iterate through those in order. 

## In 1.1, it reads the job_files and complements it with defaults if necessary
## In 1.2, it complements the global environment with any user choice specified in the "environment" of the job files. For instance, the variable "requested_procs"
## In 1.3, if the environment does not correspond to a computation cluster, or is poorly configured, scope disables submission, and input/submission files are not generated
## In 1.4 and 1.5, it finds the requested Branch and the Workflow. If they do not exist yet, they are created. 
##                 This means that you don't need to initiate the classes of the computational workflow in advance 



### Step 2:

In [10]:
## In 2.1, it finds or creates the Job using the "job_data" section in the job_file.
this_job.job_data  ## This is the job_data with which the job was created

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.branch              | <class 'str'>       | Isolated  
self.workflow            | <class 'list'>      | ['ref_hs_mol', 'ref_ls_mol']
self.setup               | <class 'str'>       | regular   
self.suffix              | <class 'str'>       | opt2      
self.keyword             | <class 'str'>       | b3lyp_opt 
self.istate              | <class 'str'>       | pbe_opt   
self.fstate              | <class 'str'>       | b3lyp_opt 
self.hierarchy           | <class 'int'>       | 2         
self.requisites          | <class 'list'>      | ['pbe_opt']
self.constrains          | <class 'list'>      | ['self']  
self.must_be_good        | <class 'bool'>      | True      
self.job_setup           | <class 'str'>       | regular   

In [11]:
## In 2.2, if the Job already exists, it checks whether the "job_data" was updated by the user

In [12]:
## In 2.3, it checks whether the "job" requisites and constrains are fulfilled. If not, the job is skipped
print(this_job.requisites, this_job.requisites_fulfilled)
print(this_job.constrains, this_job.constrains_fulfilled)

## In this case, both were fulfilled

['pbe_opt'] True
['self'] True


### Step 3:

In [13]:
## In Step 3, it finds or creates the computations, using the variable "job_setup" in the "job_data" section of the job_file:
print(this_job.job_data.job_setup)

## There are 4 options for of job_setup:
##  - "regular" or "reg":          The most common. A Job that expects one computation
##  - "displacement" or "disp":    This job expects an initial state (istate) with vibrational normal modes. It will take those VNM with negative frequencies and prepare a new initial geometry that follows the modes (eigenvectors) 
##  - "findiff":                   This job will prepare 3N geometries using the VNM of the initial state to obtain the Hessian through a "finite differences" analysis. Each geometry will be associated with a state, and with a single-point computation
##  - "rep_opt":                   This job will prepare one geometry optimization computation, but the job will require energy convergence along several successive computations. This is useful for variable-cell optimizations in Quantum Espresso

regular


In [14]:
## In Step 3.1, it checks whether the "qc_data" was updated by the user, and whether the files (input, output, submission) exist
comp = this_job.computations[0]  ## This is the first and only computation associated with this job
print(comp.qc_data)              ## This is its QC_DATA 

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.software            | <class 'str'>       | g16       
self.jobtype             | <class 'str'>       | opt       
self.functional          | <class 'str'>       | b3lyp**   
self.basis               | <class 'str'>       | def2svp   
self.is_grimme           | <class 'bool'>      | True      
self.loose_opt           | <class 'bool'>      | False     
self.tight_opt           | <class 'bool'>      | False     
self.grimme_type         | <class 'str'>       | d2        
self.fstate              | <class 'str'>       | b3lyp_opt 
self.istate              | <class 'str'>       | pbe_opt   
self.section             | <class 'str'>       | &qc_data  
self.type                | <class 'str'>       | input_data



In [ ]:
## It also checks whether "continuation" computations exist. This is a very important piece of information if the system file is ever deleted or corrupted. 
## The way this is done brings us to how file names are defined. They are handled by its own class, the Filename:
print(comp.filename)
## Below are the ITEMS that were used to construct the name of this file. The ITEMS that are employed depend on the job_type 

sys_name: ABITEM. Format: ABITEM
sou_name: ref_hs_mol. Format: ref_hs_mol
suffix: opt2. Format: opt2
run_number: 1. Format: r1



In [21]:
## Basically, the Filename class enables SCOPE to identify whether continuation computations exists.
## A "continuation" computation basically shares the same name of the original one (that is, the same items in comp.filename) except for the "run_number". This is the run_number of our computation
print(comp.run_number)
## It indicates that it is the first attempt. The run number is the "_r1" part of the filename
print(comp.inp_name)
## In this case, the computation worked at the first attempt, so there was no need for an update with "_r2". Hence: 
print(comp.has_update)  
## If otherwise, if a "continuation" computation exists in the calculations folder, SCOPE will not create it once again, but will wait to register it. 

1
ABITEM_ref_hs_mol_opt2_r1.com
False


In [22]:
## In Step 3.2, it checks whether it must submit the computation. It will do so if:
## (1) The output file does not exist. 
## (2) The computation does not have a continuation (i.e. comp.has_update == False) 
## (3) The user wants to submit computations, as specified in the job_file. Variable "want_submit" in &options section of the job file. 
##     The default is True, but reverts to False if SLURM or SGE queue management is not detected
## (4) The job is not running already. This is evaluated by the environment using the "subprocess" module with predefined commands.
##     This check can be skipped by the variable "ignore_submitted" in the &options section of the job file. Not recommended.

## Also, if the output file exists, but not the input file, the function will show a warning prompting the user to investigate.

### Step 4:

In [43]:
## In step 4, if both input and output files exist, the computation might be "REGISTERED". Registering includes Parsing the output, and many other things 

In [44]:
## Step 4.1:
## - If the computation is already registered, this step will be skipped. The output parsing takes time so this step is skipped whenever possible. 
##   This is what would happen in "Case 4" of the example in cell number 3 of this notebook (see above):

## - If not, the computation is registered (see Computation.Register() function in classes_workflow). Below is a summary of what it does:
##   - Files are checked: 
print(comp.check_files())
##   - Output-class is created:
comp.create_output()
##   - General Properties are registered
print(comp.isfinished)
print(comp.elapsed_time)
print(comp.status)
##   - Energy is registered and stored in the final state.
found, istate = comp.source.find_state(comp.qc_data.istate)
found, fstate = comp.source.find_state(comp.qc_data.fstate)
print(fstate.results["energy"])
##   - Because this is a job that implies a geometry optimization, the updated geometry is registered and stored in the final state.
print("Original coordinates of the first atom:", istate.coord[0])  ## from the initial state
print("Final coordinates of the first atom:   ", fstate.coord[0])     ## from the final state

None
True
44218.6
worked
energy: -3159.62579289 au
Original coordinates of the first atom: [3.738143, 3.85153, 13.537435]
Final coordinates of the first atom:    [3.774774, 3.779086, 13.494815]


In [46]:
## In Step 4.2 it sets up continuation computations if needed.
## In Step 4.3 it handles repetitive jobs (only if this_job.job_setup = "rep_opt")
this_job.job_setup

'regular'

In [47]:
## In Step 4.4 it handles some errors for which a simple and routine solution exists. 
## This is done only if the flag "-e" is called when executing scope_run_job.